## make point mutation

In [ ]:
from experiments.utils import modify_residues_in_cdr

src_pkl = '/home/psh/benchmark_after210930/meta/8hs2_B_C_R.pkl'
target_dir = '/home/psh/benchmark_after210930/meta_modif'
cdr_sequence = "GAGGFLRIITKFDY"

mutation_list = [[(7, "GLY")], [(8, "GLY")], [(7, "GLY"), (8, "GLY")]]

for mutations in mutation_list:
    modify_residues_in_cdr(src_pkl, target_dir, cdr_sequence, mutations=mutations)

변경: global_idx=105, new_resname=GLY
✅ 저장 완료: /home/psh/benchmark_after210930/meta_modif/8hs2_B_C_R_7GLY.pkl
변경: global_idx=106, new_resname=GLY
✅ 저장 완료: /home/psh/benchmark_after210930/meta_modif/8hs2_B_C_R_8GLY.pkl
변경: global_idx=105, new_resname=GLY
변경: global_idx=106, new_resname=GLY
✅ 저장 완료: /home/psh/benchmark_after210930/meta_modif/8hs2_B_C_R_7GLY_8GLY.pkl


### save point mutation csv file

In [ ]:
import os
import pandas as pd

# 입력
input_csv_file = '/home/psh/benchmark_after210930/meta/metadata.csv'
target_dir = '/home/psh/benchmark_after210930/meta_modif'
target_wt_pdbs = ['8hs2_B_C_R']

# 1. CSV 파일 읽기
df = pd.read_csv(input_csv_file)

# 2. target_wt_pdbs에 해당하는 행 선택
filtered_df = df[df['pdb_name'].isin(target_wt_pdbs)].copy()

# 3. target_dir 내부 파일 리스트 확인
files = os.listdir(target_dir)

# 4. target_wt_pdbs 각각에 대해 포함된 파일 개수를 세고 행 복제
duplicated_rows = []
for pdb_name in target_wt_pdbs:
    # pdb_name이 포함된 파일만 필터링
    matched_files = [f for f in files if pdb_name in f]
    print(f"{pdb_name} 관련 파일 개수: {len(matched_files)}")

    if len(matched_files) > 0:
        # 해당 pdb_name 행 가져오기
        rows = filtered_df[filtered_df['pdb_name'] == pdb_name]
        base_row = rows.iloc[0]  # 보통 하나일 것이므로 첫 행 기준
        
        # 각 파일마다 하나의 행 생성
        for fname in matched_files:
            new_row = base_row.copy()
            new_row['processed_path'] = os.path.join(target_dir, fname)
            duplicated_rows.append(new_row)

# 5. 결과 DataFrame 생성
if duplicated_rows:
    result_df = pd.DataFrame(duplicated_rows)
else:
    result_df = pd.DataFrame(columns=df.columns)

# 6. 결과 확인 및 저장
result_df.to_csv('/home/psh/benchmark_after210930/meta_modif/metadata_duplicated.csv', index=False)

8hs2_B_C_R 관련 파일 개수: 3


## visualize (save pse file)

In [9]:
import glob
import os 

from pymol import cmd

cmd.reinitialize()

## get point mutation 
root_dir = "/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2/2025-10-14_00-20-37/epoch=63-step=91712_copy/8hs2_point_mutation"
align_num = 10

files = []
mutations = []
for mutation in os.listdir(root_dir):
     if 'config' in mutation or '.png' in mutation:
         continue
     mut_dir = os.path.join(root_dir, mutation)

     count=0  
     for sample in os.listdir(mut_dir):
        if 'sample' not in sample:
            continue
        count += 1
        sample_path = os.path.join(mut_dir, sample, "sample_1.pdb")
        mutations.append(mutation)
        files.append(sample_path)
        if count == align_num:
             break
print(files)
for i, f in enumerate(files):
    mutation = mutations[i].replace('.pkl', '')
    cmd.load(f, f'{mutation}_sample_{i}')

########################################################################################################################################################################
## get wt prediction 
root_dir = "/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2/2025-10-14_00-20-37/epoch=63-step=91712_copy/WT/8hs2_B_C_R"
title = root_dir.split('/')[-1]

files = []
count=0  

for sample in os.listdir(root_dir):
    if 'config' in sample or '.png' in sample:
        continue
    count+=1
    sample_dir = os.path.join(root_dir, sample)
    sample_path = os.path.join(sample_dir, "sample_1.pdb")
    files.append(sample_path)

    if count == align_num:
        break
        
# 나머지 파일 로드 및 정렬
for i, f in enumerate(files):
    cmd.load(f, f'sample_{i}')

# 색상 랜덤 지정
cmd.util.cbc()
# H3 루프 선택 및 강조 (필요 시 수정)
cmd.save(f"/home/psh/protein-frame-flow/point_mutation/{title}_aligned_all.pse")

['/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2/2025-10-14_00-20-37/epoch=63-step=91712_copy/8hs2_point_mutation/8hs2_B_C_R_7GLY.pkl/sample_0/sample_1.pdb', '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2/2025-10-14_00-20-37/epoch=63-step=91712_copy/8hs2_point_mutation/8hs2_B_C_R_7GLY.pkl/sample_1/sample_1.pdb', '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2/2025-10-14_00-20-37/epoch=63-step=91712_copy/8hs2_point_mutation/8hs2_B_C_R_7GLY.pkl/sample_2/sample_1.pdb', '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2/2025-10-14_00-20-37/epoch=63-step=91712_copy/8hs2_point_mutation/8hs2_B_C_R_7GLY.pkl/sample_3/sample_1.pdb', '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2/2025-10-14_00-20-37/epoch=63-step=91712_copy/8hs2_point_mutation/8hs2_B_C_R_7GLY.pkl/sample_4/sample_1.pdb', '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2/2025-10-14_00-20